# Memory Experiment — GALA Codes

Group-Action Lifts with Active orthogonality (Yang, Duckering, Dua,
arXiv:2608.07431). GALA generalises the Kasai construction: the block-circulant
parents are the same shape, but each lift entry lives in the group ring
`F_2[H_k x C_m]` instead of being a single affine permutation, giving
`n = L * k * m` data qubits.

The small non-abelian `H_k` fixes the orthogonality pattern; the abelian `C_m`
supplies the shift symmetries that become AOD move schedules.

| preset | n | k | d | lift |
|---|---|---|---|---|
| `gala_132_30_12` | 132 | 30 | 12 | `C_11` |
| `gala_136_36_8` | 136 | 36 | 8 | `C_17` (polynomial) |
| `gala_168_42_12` | 168 | 42 | 12 | `C_3 x C_7` (polynomial) |
| `gala_192_40_12` | 192 | 40 | 12 | `C_16` |
| `gala_228_46_12` | 228 | 46 | 12 | `C_19` |
| `gala_576_104_12` | 576 | 104 | 12 | `S_3 x C_16` |
| `gala_576_292_8` | 576 | 292 | 8 | `S_3 x C_2 x C_8` (rate 1/2) |
| `gala_720_364_10` | 720 | 364 | 10 | `S_3 x C_4 x C_5` (rate 1/2) |
| `gala_1056_532_12` | 1056 | 532 | 12 | `S_4 x C_2 x C_11` (rate 1/2) |

Decoding uses plain BP (`ldpc-bp`) through `SimulationPipeline`, on the
Z-detector-only DEM — same setup as `memory_kasai.ipynb`.

**Runtime:** the (n, k) replication below is seconds. Circuit builds scale with
`rounds * n`, so `gala_132_30_12` is interactive while the 576+ presets take
minutes per build.

**Scope:** code construction + syndrome extraction + experiment-level
observables. `GalaCode` reports `num_logicals` but exposes no explicit logical
operator representatives (`logical_ops_available = False`), as for Kasai codes.


In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.ir.qec_system import QECSystem
from lightstim.noise.config import NoiseConfig
from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.gala_code import (
    GALA_CODE_PRESETS, GalaCode, GalaCodeExtractionBlock,
)
from lightstim.simulation.decoder_backend import DecoderConfig, SimulationPipeline
from lightstim.simulation.decoder_backend.registry import list_decoders

assert "ldpc-bp" in list_decoders(), "pip install ldpc to register the plain-BP decoder"

## Replicate published (n, k)

Check every preset against arXiv:2608.07431, and verify the CSS condition
`H_X H_Z^T = 0` directly.

In [2]:
rows = []
for name in sorted(GALA_CODE_PRESETS):
    preset = GALA_CODE_PRESETS[name]
    code_obj = GalaCode.from_preset(name)
    rows.append({
        "preset": name,
        "L": code_obj.L, "J": code_obj.J,
        "lift": f"k={code_obj.degree}, C={code_obj.cyclic}",
        "n": code_obj.n_data, "k": code_obj.num_logicals,
        "expected_n": preset["expected_n"], "expected_k": preset["expected_k"],
        "d_published": preset["expected_d"],
        "weight": code_obj.stabilizer_weight,
        "css_ok": code_obj.check_css_orthogonality(),
    })

df_nk = pd.DataFrame(rows)
assert (df_nk["n"] == df_nk["expected_n"]).all()
assert (df_nk["k"] == df_nk["expected_k"]).all()
assert df_nk["css_ok"].all()
df_nk

,preset,L,J,lift,n,k,expected_n,expected_k,d_published,weight,css_ok
0,gala_1008_510_10,12,3,"k=3, C=(2, 2, 7)",1008,510,1008,510,10,12,True
1,gala_1056_532_12,12,3,"k=4, C=(2, 11)",1056,532,1056,532,12,12,True
2,gala_132_30_12,12,5,"k=1, C=(11,)",132,30,132,30,12,12,True
3,gala_136_34_12,8,3,"k=1, C=(17,)",136,34,136,34,12,12,True
4,gala_136_36_8,8,3,"k=1, C=(17,)",136,36,136,36,8,12,True
5,gala_168_42_12,8,3,"k=1, C=(3, 7)",168,42,168,42,12,12,True
6,gala_192_40_12,12,5,"k=1, C=(16,)",192,40,192,40,12,12,True
7,gala_228_46_12,12,5,"k=1, C=(19,)",228,46,228,46,12,12,True
8,gala_576_104_12,12,5,"k=3, C=(16,)",576,104,576,104,12,12,True
9,gala_576_292_8,12,3,"k=3, C=(2, 8)",576,292,576,292,8,12,True


### A note on the orthogonality criterion

Definition 11 asks that `[F_i, G_j] = 0` for every pair in the band `Gamma_J`.
That is *sufficient*, not necessary: orthogonality only needs the sums
`Psi_r = sum_u [F_u, G_{r-u}]` to vanish for `r < J`, and individual
commutators may cancel within a sum.

Several published rate-1/2 instances do exactly that, so
`validate_required_commutativity()` (the strict per-pair test) returns `False`
for them while `check_css_orthogonality()` — the authoritative test — passes.

In [3]:
strict = {name: GalaCode.from_preset(name, compute_k=False)
          for name in ["gala_132_30_12", "gala_576_292_8"]}
for name, c in strict.items():
    print(f"{name:18s} strict Definition-11 pairs: {c.validate_required_commutativity()!s:5s} "
          f"| H_X H_Z^T = 0: {c.check_css_orthogonality()}")

gala_132_30_12     strict Definition-11 pairs: True  | H_X H_Z^T = 0: True
gala_576_292_8     strict Definition-11 pairs: False | H_X H_Z^T = 0: True


## Configuration

Defaults mirror `memory_kasai.ipynb`: idling noise off, `z_only = True` so only
Z-ancilla measurements emit detectors (essential for plain BP), and
non-converged shots heralded as logical errors.

In [4]:
PRESET      = "gala_132_30_12"   # smallest instance; interactive
P_VALUES    = [1e-3]
ROUNDS      = 12                 # ~ d for this code
Z_ONLY      = True
MAX_SHOTS   = 2_000
MAX_ERRORS  = 100
NUM_WORKERS = 4
BATCH_SIZE  = 100

BP_PARAMS = {
    "max_iter": 200,
    "bp_method": "minimum_sum",
    "ms_scaling_factor": 0.0,   # 0 = ldpc's dynamic scaling
    "schedule": "serial",       # parallel flooding oscillates on these DEMs
}

## Build the memory circuit

In [5]:
def build_circuit(preset, p, rounds=ROUNDS, z_only=Z_ONLY,
                  se_block=GalaCodeExtractionBlock):
    code_obj = GalaCode.from_preset(preset)
    system = QECSystem()
    system.add_patch(code_obj, name=preset)
    noise = NoiseConfig(p_idle=0.0, p_1q=p, p_2q=p, p_meas=p, p_reset=p)
    circuit = MemoryExperiment(
        qec_system=system,
        extraction_block_class=se_block,
        rounds=rounds,
        noise_params=noise,
        noise_model="circuit_level",
        basis="Z",
        z_only=z_only,
    ).build()
    return circuit, code_obj

circuit, code_obj = build_circuit(PRESET, P_VALUES[0])
print(f"qubits={circuit.num_qubits}  detectors={circuit.num_detectors}  "
      f"observables={circuit.num_observables}  k={code_obj.num_logicals}")

qubits=242  detectors=715  observables=30  k=30


## Decode with plain BP (tier-1 style)

In [6]:
pipeline = SimulationPipeline(
    decoder_config=DecoderConfig("ldpc-bp", backend="cpu", params=BP_PARAMS,
                                 on_decode_failure="error"),
    max_shots=MAX_SHOTS,
    max_errors=MAX_ERRORS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    print_progress=True,
)

results = []
for p in P_VALUES:
    circuit, code_obj = build_circuit(PRESET, p)
    task = {"code": PRESET, "p": p, "rounds": ROUNDS, "z_only": Z_ONLY,
            "decoder_name": "ldpc-bp"}
    t0 = time.perf_counter()
    stats = pipeline.run(circuit, task)
    ler = stats.logical_error_rate
    # per-round conversion (eq. 7 of arXiv:2510.14060)
    ler_round = (1 - (1 - 2 * ler) ** (1 / ROUNDS)) / 2 if ler < 0.5 else 0.5
    results.append({**task, "shots": stats.shots, "errors": stats.errors,
                    "ler_shot": ler, "ler_round": ler_round,
                    "seconds": time.perf_counter() - t0})
    print(f"p={p:.2e}: LER/shot={ler:.3e}  LER/round={ler_round:.3e}  "
          f"({stats.errors}/{stats.shots})")

pd.DataFrame(results)

shots=0 kept=0 errors=0 LER=0.00e+00±-- elapsed=0.1s ETA=--

shots=0 kept=0 errors=0 LER=0.00e+00±-- elapsed=10.1s ETA=--

shots=0 kept=0 errors=0 LER=0.00e+00±-- elapsed=20.1s ETA=--

shots=0 kept=0 errors=0 LER=0.00e+00±-- elapsed=30.1s ETA=--

shots=100 kept=100 errors=1 LER=1.00e-02±1.95e-02 elapsed=40.2s ETA=12m43s

...

shots=1,800 kept=1,800 errors=50 LER=2.78e-02±7.59e-03 elapsed=290.6s ETA=32s

shots=1,900 kept=1,900 errors=54 LER=2.84e-02±7.47e-03 elapsed=300.6s ETA=16s

final shots=2,000 kept=2,000 errors=60 LER=3.00e-02±7.48e-03 elapsed=306.6s ETA=0s


p=1.00e-03: LER/shot=3.000e-02  LER/round=2.572e-03  (60/2000)


,code,p,rounds,z_only,decoder_name,shots,errors,ler_shot,ler_round,seconds
0,gala_132_30_12,0.001,12,True,ldpc-bp,2000,60,0.03,0.002572,306.650349


The paper reports LER below `1e-8` for `[[132,30,12]]` memory at
`p = 1e-3` using its full hierarchical decoder (BP -> relay-BP -> MLE) with a
3.1 ms SE cycle. Plain BP alone, as configured here, is the tier-1 stage only
and has a much higher floor; chain it with relay-BP (see the
`DecoderConfig` list form in `memory_kasai.ipynb`) to approach the published
numbers.